In [2]:
# Importando as bibliotecas

import pandas as pd 
import numpy as np
import os
from datetime import datetime, date
from google.cloud import bigquery
from dotenv import load_dotenv
import pyarrow


# Carregando variáveis de ambiente
load_dotenv()

True

In [3]:
# Imprtando dados de 2016_01

df_2016_01 = pd.read_csv("/home/danielpedro/Engenharia de Dados/combustivel_preco2015_2025/dataset/ca-2016-01.csv", sep=';')

In [4]:
df_2016_02 = pd.read_csv("/home/danielpedro/Engenharia de Dados/combustivel_preco2015_2025/dataset/ca-2016-02.csv", sep=';')

In [5]:
# Concatenando os dataframes de 2016_01 a 2016_02
df_2016 = pd.concat((df_2016_01, df_2016_02))

In [6]:
df_2016.info()

<class 'pandas.core.frame.DataFrame'>
Index: 975857 entries, 0 to 488959
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   Regiao - Sigla     975857 non-null  object
 1   Estado - Sigla     975857 non-null  object
 2   Municipio          975857 non-null  object
 3   Revenda            975857 non-null  object
 4   CNPJ da Revenda    975857 non-null  object
 5   Nome da Rua        975857 non-null  object
 6   Numero Rua         975286 non-null  object
 7   Complemento        250979 non-null  object
 8   Bairro             972105 non-null  object
 9   Cep                975857 non-null  object
 10  Produto            975857 non-null  object
 11  Data da Coleta     975857 non-null  object
 12  Valor de Venda     975857 non-null  object
 13  Valor de Compra    419788 non-null  object
 14  Unidade de Medida  975857 non-null  object
 15  Bandeira           975857 non-null  object
dtypes: object(16)
memory usag

In [7]:
# Filtro aplicado na extração: apenas registros do estado de Pernambuco (UF == "PE")
# Decisão: reduzir volume de armazenamento e escopo do projeto
df_2016_pe = df_2016[df_2016["Estado - Sigla"] == "PE"]

In [8]:
# Ajustndo o tipo de dado da coluna "Data da Coleta" para datetime
df_2016_pe["Data da Coleta"] = pd.to_datetime(df_2016_pe["Data da Coleta"], errors="coerce")


/tmp/ipykernel_993/3168891963.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2016_pe["Data da Coleta"] = pd.to_datetime(df_2016_pe["Data da Coleta"], errors="coerce")


In [9]:
# Ajustando o tipo de dado da coluna "Valor de Venda" e "Valor de Compra" para numérico
df_2016_pe["Valor de Venda"] = pd.to_numeric(df_2016_pe["Valor de Venda"], errors="coerce")
df_2016_pe["Valor de Compra"] = pd.to_numeric(df_2016_pe["Valor de Compra"], errors="coerce")


/tmp/ipykernel_993/2928484231.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2016_pe["Valor de Venda"] = pd.to_numeric(df_2016_pe["Valor de Venda"], errors="coerce")
/tmp/ipykernel_993/2928484231.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2016_pe["Valor de Compra"] = pd.to_numeric(df_2016_pe["Valor de Compra"], errors="coerce")


In [10]:
df_2016_pe.head()

,Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
3905,NE,PE,ABREU E LIMA,COSTA DOURADA COMBUSTÍVEIS E LUBRIFICANTES LTDA.,06.053.230/0002-86,AVENIDA D,S/N,NaN,CAETES,53540-000,GASOLINA,2016-04-01,NaN,NaN,R$ / litro,FEDERAL
3906,NE,PE,ABREU E LIMA,COSTA DOURADA COMBUSTÍVEIS E LUBRIFICANTES LTDA.,06.053.230/0002-86,AVENIDA D,S/N,NaN,CAETES,53540-000,ETANOL,2016-04-01,NaN,NaN,R$ / litro,FEDERAL
3907,NE,PE,ABREU E LIMA,COSTA DOURADA COMBUSTÍVEIS E LUBRIFICANTES LTDA.,06.053.230/0002-86,AVENIDA D,S/N,NaN,CAETES,53540-000,DIESEL S10,2016-04-01,NaN,NaN,R$ / litro,FEDERAL
3908,NE,PE,ABREU E LIMA,AUTO POSTO CABRAL COMBUSTIVEIS LTDA,08.131.708/0001-93,AVENIDA DUQUE DE CAXIAS,1162,NaN,BOA ESPERANCA,53580-020,GASOLINA,2016-04-01,NaN,NaN,R$ / litro,BRANCA
3909,NE,PE,ABREU E LIMA,AUTO POSTO CABRAL COMBUSTIVEIS LTDA,08.131.708/0001-93,AVENIDA DUQUE DE CAXIAS,1162,NaN,BOA ESPERANCA,53580-020,ETANOL,2016-04-01,NaN,NaN,R$ / litro,BRANCA


In [11]:
# Configurando credenciais

# Caminho do arquivo de credenciais GCP
credencial = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

# Identificador do projeto no GCP
project_id = os.getenv("PROJECT_ID")

# Tabela Bronze no BigQuery
table_id = os.getenv("BRONZE_2016")

# Inicialização do cliente BigQuery

# Define a variavel de ambiente
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = credencial

# Cria o cliente Bigquery ao projeto configurado
client = bigquery.Client(project=project_id)

In [12]:
# Criação e carga da tabela BRONZE

# Configuração do job de carga no BigQuery
# WRITE_TRUNCATE garante que a tabela Bronze seja sempre sobrescrita,
# mantendo apenas o retrato mais recente dos dados de origem
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

# Envia o DataFrame para o BigQuery,
# criando a tabela automaticamente caso ainda não exista
job = client.load_table_from_dataframe(
    df_2016_pe,
    table_id,
    job_config=job_config
)

# Aguarda a finalização do job para garantir consistência da carga
job.result()

# Confirmação visual de sucesso da operação
print("✅ Tabela Bronze criada e dados carregados com sucesso")

/home/danielpedro/anaconda3/envs/combustivel_projeto/lib/python3.11/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


✅ Tabela Bronze criada e dados carregados com sucesso
